In [ ]:
from theia.detection.pcl import PclDetector
from theia.test_data import load_pcl_example

sensors, trajcetories, grid = load_pcl_example()
trajectory = trajcetories[0]
RCS = trajectory.cross_section_model.rcs

In [ ]:
from theia.detection.pcl import pcl_track_init_update_masks

detector = PclDetector()
track_init_mask, track_update_mask = pcl_track_init_update_masks(
    detector,
    sensors,
    grid,
    RCS,
)

In [ ]:
import folium

from theia.mapping import RadarMap
from theia.util import mask_to_polygon


map = RadarMap(
    sensors={f"Sensor {sensor.id}": sensor for sensor in sensors},
    trajectories={"Target": trajectory},
).to_map()
map.location = (sensors[0].receiver.lat, sensors[0].receiver.lon)

polygons = mask_to_polygon(
    track_init_mask[:, :, 0],
    grid.latitude_values[0],
    grid.latitude_values[1] - grid.latitude_values[0],
    grid.longitude_values[0],
    grid.longitude_values[1] - grid.longitude_values[0],
)
for polygon in polygons:
    folium.GeoJson(polygon, fillColor="red", color="red").add_to(map)

polygons = mask_to_polygon(
    track_update_mask[:, :, 0],
    grid.latitude_values[0],
    grid.latitude_values[1] - grid.latitude_values[0],
    grid.longitude_values[0],
    grid.longitude_values[1] - grid.longitude_values[0],
)
for polygon in polygons:
    folium.GeoJson(polygon, fillColor="blue", color="blue").add_to(map)
map

In [ ]:
import numpy as np

from theia.detection.pcl import PclDetector
from theia.export_paraview import ParaviewExporter, PointOfInterest


pois: list[PointOfInterest] = []
poi_id = 0
for sensor in pcl_sensors:
    pois.append(
        PointOfInterest(
            id=poi_id,
            label=f"Rx {sensor.receiver.id}",
            type="Rx",
            lat=sensor.receiver.lat,
            lon=sensor.receiver.lon,
            alt=sensor.receiver.alt,
        )
    )
    poi_id += 1
    pois.append(
        PointOfInterest(
            id=poi_id,
            label=f"Tx {sensor.transmitter.id}",
            type="Tx",
            lat=sensor.transmitter.lat,
            lon=sensor.transmitter.lon,
            alt=sensor.transmitter.alt,
        )
    )
    poi_id += 1

exporter = ParaviewExporter(
    lat_min,
    lat_max,
    lat_res,
    lon_min,
    lon_max,
    lon_res,
)

sensor = pcl_sensors[0]

detector = PclDetector()

exporter.export_terrain(
    "test",
    # {
    #     "min detectable RCS": lambda *p: detector.minimum_detectable_rcs_vector(
    #         sensor.receiver, sensor.transmitter, np.array(p).reshape((1, 3))
    #     )
    # },
)
exporter.export_pois(pois, "pois.csv")